# 06 - Guided Support-Router Exercise

This notebook combines the repository concepts into one local workflow. You will estimate token pressure, retrieve lesson candidates, make a confidence decision, and explain risks without calling an external LLM.


## Concept Context

A support router is a compact example of an LLM-adjacent system. Even without calling an external model, it must estimate input size, account for retrieved context, rank candidate modules, choose a confidence policy, and explain risk. Those are the same decisions that appear inside larger RAG assistants.

The exercise is intentionally local and auditable. Every intermediate value is visible so the final decision can be explained instead of treated as a black box.

## Objectives

- Inspect source data and support questions.
- Estimate token count and request cost.
- Retrieve top-k lesson candidates.
- Add a confidence threshold.
- Write a stakeholder explanation of limitations.


```mermaid
flowchart LR
    A[Support question] --> B[Count tokens]
    B --> C[Estimate cost]
    A --> D[Retrieve top-k lessons]
    D --> E[Apply threshold]
    E --> F[Use results or fallback]
    F --> G[Explain risks]
```


## End-To-End Thinking

The support router is deliberately small, but it touches the main parts of an LLM application: input size, cost, retrieval, confidence, and explanation. The point is not to build a polished product in one notebook. The point is to make every decision visible enough to debug.

When you adapt this pattern, keep the intermediate values. If a user receives a poor answer, you will want to know the question tokens, retrieved candidates, scores, threshold decision, and fallback path.


In [ ]:
# The router helpers are notebook-local so the exercise stays self-contained.
# Keep these intermediate values visible when adapting the workflow to a real project.
import csv
import math
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

TOKEN_PATTERN = re.compile(
    r"""
    [A-Za-z]+(?:'[A-Za-z]+)?  # words and contractions
    |\d+(?:\.\d+)?           # integers and decimals
    |[^\w\s]                 # punctuation and symbols
    """,
    re.VERBOSE,
)


@dataclass(frozen=True)
class TokenizationResult:
    text: str
    tokens: list[str]

    @property
    def count(self) -> int:
        return len(self.tokens)


def simple_tokenize(text: str) -> list[str]:
    """Split text into approximate tokens for transparent local practice."""
    return TOKEN_PATTERN.findall(text)


def inspect_text(text: str) -> TokenizationResult:
    return TokenizationResult(text=text, tokens=simple_tokenize(text))


def count_tokens(text: str) -> int:
    return len(simple_tokenize(text))


def token_budget_remaining(
    context_window: int, prompt_tokens: int, reserved_output_tokens: int
) -> int:
    return context_window - prompt_tokens - reserved_output_tokens


@dataclass(frozen=True)
class CostEstimate:
    input_tokens: int
    output_tokens: int
    input_cost: float
    output_cost: float

    @property
    def total_cost(self) -> float:
        return self.input_cost + self.output_cost


def estimate_request_cost(
    input_tokens: int,
    output_tokens: int,
    input_price_per_million: float,
    output_price_per_million: float,
) -> CostEstimate:
    input_cost = input_tokens / 1_000_000 * input_price_per_million
    output_cost = output_tokens / 1_000_000 * output_price_per_million
    return CostEstimate(input_tokens, output_tokens, input_cost, output_cost)


def format_usd(value: float) -> str:
    return f"${value:.6f}"


SparseVector = dict[str, float]

STOPWORDS = {
    "a",
    "an",
    "and",
    "are",
    "as",
    "be",
    "by",
    "can",
    "do",
    "does",
    "for",
    "from",
    "how",
    "i",
    "in",
    "into",
    "is",
    "it",
    "me",
    "my",
    "of",
    "or",
    "should",
    "the",
    "this",
    "to",
    "what",
    "when",
    "whether",
    "why",
    "with",
}


@dataclass(frozen=True)
class EmbeddedCorpus:
    records: list[dict[str, str]]
    text_field: str
    idf: dict[str, float]
    vectors: list[SparseVector]


def content_tokens(text: str) -> list[str]:
    tokens = [token.lower() for token in simple_tokenize(text)]
    return [
        token
        for token in tokens
        if token not in STOPWORDS and any(char.isalnum() for char in token)
    ]


def combine_text_fields(record: dict[str, str], fields: list[str]) -> str:
    missing = [field for field in fields if field not in record]
    if missing:
        raise ValueError(f"Missing expected fields: {missing}")
    return " ".join(record.get(field, "") for field in fields)


def fit_idf(texts: list[str]) -> dict[str, float]:
    document_count = len(texts)
    document_frequency: Counter[str] = Counter()
    for text in texts:
        document_frequency.update(set(content_tokens(text)))
    return {
        token: math.log((1 + document_count) / (1 + frequency)) + 1
        for token, frequency in document_frequency.items()
    }


def vectorize(text: str, idf: dict[str, float]) -> SparseVector:
    tokens = content_tokens(text)
    if not tokens:
        return {}
    counts = Counter(tokens)
    total = len(tokens)
    vector = {
        token: (count / total) * idf.get(token, 0.0)
        for token, count in counts.items()
        if token in idf
    }
    norm = math.sqrt(sum(value * value for value in vector.values()))
    if norm == 0:
        return vector
    return {token: value / norm for token, value in vector.items()}


def embed_corpus(records: list[dict[str, str]], text_field: str) -> EmbeddedCorpus:
    if not records:
        raise ValueError("Cannot embed an empty corpus.")
    texts = [record[text_field] for record in records]
    idf = fit_idf(texts)
    vectors = [vectorize(text, idf) for text in texts]
    return EmbeddedCorpus(
        records=records, text_field=text_field, idf=idf, vectors=vectors
    )


def cosine_similarity(left: SparseVector, right: SparseVector) -> float:
    if not left or not right:
        return 0.0
    shared = set(left).intersection(right)
    return sum(left[token] * right[token] for token in shared)


def similarity_scores(corpus: EmbeddedCorpus, query: str) -> list[float]:
    query_vector = vectorize(query, corpus.idf)
    return [cosine_similarity(query_vector, vector) for vector in corpus.vectors]


@dataclass(frozen=True)
class SearchResult:
    rank: int
    score: float
    lesson_id: str
    title: str
    module: str
    description: str


class SemanticSearchIndex:
    def __init__(self, corpus: EmbeddedCorpus):
        self.corpus = corpus

    @classmethod
    def from_csv(
        cls, path: str | Path, text_fields: list[str] | None = None
    ) -> "SemanticSearchIndex":
        with Path(path).open(newline="", encoding="utf-8") as file:
            records = list(csv.DictReader(file))
        fields = text_fields or ["title", "module", "description"]
        for record in records:
            record["search_text"] = combine_text_fields(record, fields)
        return cls(embed_corpus(records, text_field="search_text"))

    def search(self, query: str, k: int = 3) -> list[SearchResult]:
        scores = similarity_scores(self.corpus, query)
        ranked_indices = sorted(
            range(len(scores)), key=lambda index: scores[index], reverse=True
        )[:k]
        results = []
        for rank, index in enumerate(ranked_indices, start=1):
            row = self.corpus.records[index]
            results.append(
                SearchResult(
                    rank=rank,
                    score=float(scores[index]),
                    lesson_id=row["lesson_id"],
                    title=row["title"],
                    module=row["module"],
                    description=row["description"],
                )
            )
        return results


def relevant_in_top_k(
    results: list[SearchResult], expected_topic: str, k: int = 3
) -> bool:
    expected = expected_topic.lower().strip().replace("_", " ")
    for result in results[:k]:
        searchable = (
            " ".join(
                [result.lesson_id, result.title, result.module, result.description]
            )
            .lower()
            .replace("_", " ")
        )
        if expected in searchable:
            return True
    return False

## Task 1: Inspect The Data

Start by loading the course catalog and support questions. Before building anything, check whether the data contains the fields you need.


In [ ]:
# Load local CSV data; keeping data on disk makes the examples reproducible.
catalog_path = Path("data") / "course_catalog.csv"
questions_path = Path("data") / "support_questions.csv"
with catalog_path.open(newline="", encoding="utf-8") as file:
    catalog = list(csv.DictReader(file))
with questions_path.open(newline="", encoding="utf-8") as file:
    questions = list(csv.DictReader(file))
print(f"Catalog lessons: {len(catalog)}")
print(f"Support questions: {len(questions)}")
print("Catalog columns:", list(catalog[0]))
print("Question columns:", list(questions[0]))

## Task 1 Reference Answer

The loaded CSV files should expose the expected schemas: `lesson_id`, `title`, `module`, and `description` for the catalog; `question_id`, `question`, `expected_topic`, and `difficulty` for support questions. If any of those columns are missing, the router cannot evaluate retrieval quality reliably.

## Task 2: Configure The Router

The constants below are simple product assumptions: three retrieved lesson descriptions, a short answer, and example token prices.


## Router Decision Flow

```mermaid
flowchart TD
    A[New support question] --> B[Count question tokens]
    B --> C[Estimate cost with retrieved context]
    A --> D[Search course catalog]
    D --> E[Inspect top-3 lessons]
    E --> F{Top score above threshold?}
    F -- Yes --> G[Use retrieved candidates]
    F -- No --> H[Fallback or ask clarification]
    G --> I[Explain limitations]
    H --> I
```

A real support assistant would likely add authentication, logging, citations, user feedback, and monitoring. Those are intentionally outside this local exercise.


In [ ]:
# Build the index once, then keep routing decisions consistent across questions.
index = SemanticSearchIndex.from_csv(catalog_path)
RETRIEVED_CONTEXT_TOKENS = 3 * 120
EXPECTED_OUTPUT_TOKENS = 180
INPUT_PRICE_PER_MILLION = 0.50
OUTPUT_PRICE_PER_MILLION = 1.50
CONFIDENCE_THRESHOLD = 0.15

## Task 2 Reference Answer

The constants define product assumptions, not universal truths. Retrieved context increases input tokens, expected output tokens reserve space for the answer, token prices make cost visible, and the confidence threshold determines when the system should fall back.

## Task 3: Define A Routing Helper

The helper keeps each decision visible: token count, cost, retrieval result, and confidence decision.


In [ ]:
# Keep routing logic in one helper so every decision exposes the same evidence.
def route_question(question: str) -> dict:
    question_tokens = count_tokens(question)
    input_tokens = question_tokens + RETRIEVED_CONTEXT_TOKENS
    cost = estimate_request_cost(
        input_tokens=input_tokens,
        output_tokens=EXPECTED_OUTPUT_TOKENS,
        input_price_per_million=INPUT_PRICE_PER_MILLION,
        output_price_per_million=OUTPUT_PRICE_PER_MILLION,
    )
    results = index.search(question, k=3)
    decision = (
        "fallback"
        if results[0].score < CONFIDENCE_THRESHOLD
        else "use retrieved candidates"
    )
    return {
        "question": question,
        "question_tokens": question_tokens,
        "input_tokens": input_tokens,
        "estimated_cost": cost,
        "results": results,
        "decision": decision,
    }

## Task 3 Reference Answer

The helper should return a structured decision that includes the original question, approximate token count, estimated request cost, top retrieval results, top score, and final action. That structure makes the routing decision auditable.

## Task 4: Route Three Questions

Use one tokenization question, one embedding/search question, and one unrelated question that should trigger fallback behavior.


In [ ]:
# Route representative in-scope and out-of-scope questions through the same helper.
sample_questions = [
    "How do I know if my prompt will fit into the context window?",
    "Can I search course notes by meaning instead of exact keywords?",
    "How do I repair a bicycle chain?",
]
for question in sample_questions:
    routed = route_question(question)
    print(f"\nQuestion: {routed['question']}")
    print(f"Question tokens: {routed['question_tokens']}")
    print(f"Estimated input tokens with retrieved context: {routed['input_tokens']}")
    print(f"Estimated request cost: {format_usd(routed['estimated_cost'].total_cost)}")
    print(f"Decision: {routed['decision']}")
    for result in routed["results"]:
        print(f"  {result.rank}. {result.title} | score={result.score:.3f}")

## Task 4 Reference Answer

The two in-scope questions should retrieve course modules related to context windows or semantic search. The unrelated repair question should be treated carefully: even if it receives a nearest neighbor, the decision should fall back when the top score is below the chosen threshold.

## Task 5: Tune The Threshold

Try multiple thresholds and inspect what changes. A threshold that catches unrelated questions can still reject useful questions if it is too high.


In [ ]:
# Compare several thresholds to understand the tradeoff between coverage and fallback.
thresholds = [0.10, 0.15, 0.20, 0.30]
for threshold in thresholds:
    print(f"\nThreshold: {threshold}")
    for question in sample_questions:
        top = index.search(question, k=3)[0]
        decision = "fallback" if top.score < threshold else "use"
        print(f"  {decision:8} | {top.score:.3f} | {question}")

## Task 5 Reference Answer

A useful threshold separates weak unrelated matches from normal in-scope questions. If increasing the threshold rejects valid questions, the setting is too strict for support routing. If decreasing it accepts unrelated questions, the setting is too permissive.

## Interpreting Threshold Experiments

When you test thresholds, do not look only at the unrelated question. Also check whether useful questions are accidentally rejected. A threshold that catches every unrelated question may still be too strict if it rejects ordinary support questions.

Record your threshold choice with a short justification. In production, that justification helps future developers understand whether the threshold was chosen for safety, answer rate, cost, or another product priority.


## Task 6: Explain The System

A stakeholder explanation should state what the system does, why token count matters, why retrieved lessons help, and why the result can still be wrong.


In [ ]:
# Draft the non-technical explanation as an output artifact, not as hidden reasoning.
stakeholder_summary = """
This helper estimates the size and approximate cost of a support question, retrieves the most similar lesson descriptions, and flags low-confidence matches for fallback. It can help route questions to useful course material, but it can still be wrong because similarity scores are relative to the indexed documents and the simple vectorizer may miss meaning or over-weight shared words. In a real application, I would monitor top-k relevance, low-confidence fallback rate, and user feedback on bad matches.
"""
print(stakeholder_summary.strip())

## Task 6 Reference Answer

A clear stakeholder explanation should mention the system input, the retrieval step, the cost estimate, the confidence threshold, and the reason fallback exists. It should avoid claiming that similarity search proves correctness.

## Worked Solution Pattern

A complete solution should include:

- three new support questions;
- a token and cost table;
- top-3 retrieval results for each question;
- a chosen threshold with justification;
- a short risk explanation for a non-technical stakeholder.


## What To Inspect

Focus on the decision, not only the top title. A retrieved lesson can look plausible while the score is weak or the query is unrelated.


## Practice

Edit `new_question` and rerun the cell. Then change `CONFIDENCE_THRESHOLD` above and rerun Tasks 4 and 5.


## Extension Ideas

- Add a `source_url` column to the catalog and print citations with each result.
- Add five new support questions and compute recall@3 again.
- Add a separate fallback message for unrelated questions versus ambiguous course questions.
- Log each routed question to a small CSV file, then summarize fallback rate and average estimated cost.
- Replace the local vectorizer with a real embedding model only after the local evaluation flow is clear.

These extensions should preserve the same debugging habit: every answer should be traceable back to input, retrieved context, score, threshold, and decision.


In [ ]:
# Editable practice question for testing one additional routing decision.
new_question = (
    "Why does semantic search still return something for an unrelated question?"
)
routed = route_question(new_question)
print(f"Decision: {routed['decision']}")
for result in routed["results"]:
    print(f"{result.rank}. {result.title} | score={result.score:.3f}")

## Practice Reference Answer

A complete practice run should print the routing decision, token estimate, cost estimate, and top-3 module candidates for the edited question. If changing `CONFIDENCE_THRESHOLD` flips a decision, record why the stricter or looser threshold is appropriate. The answer should connect the decision to evidence: score, retrieved module titles, and whether the question is in scope.

## Submission Checklist

Before finishing, make sure you can explain:

- the difference between a token and a word;
- why retrieved context changes cost;
- why embeddings help with semantic search;
- why semantic search can return weak matches;
- which threshold you would choose and why;
- one risk to monitor before using this inside an LLM application.
